# Clase 10 — Práctica: TensorFlow Extended (TFX)

## Qué vas a practicar

- Añadir validación de datos con TensorFlow Data Validation (TFDV) a un pipeline de TensorFlow Extended (TFX) ya construido
- Cambiar la arquitectura del modelo que entrena el componente `Trainer`, sin tocar el resto del pipeline

Este notebook parte del pipeline mínimo que construiste en la clase (`ExampleGen` + `Trainer` + `Pusher`) sobre el conjunto de datos Palmer Penguins. Si no lo recuerdas bien, repasa `clase10_tfx_produccion.ipynb` antes de continuar.

**Antes de empezar:** la instalación de `tfx` puede tardar varios minutos en Colab. Ejecuta la celda de instalación y ten paciencia.


## Preparación

Instalamos `tfx`, comprobamos versiones y descargamos el conjunto de datos, exactamente igual que en el notebook de teoría.


In [ ]:
try:
    import colab
    !pip install --upgrade pip -q
except ImportError:
    pass

!pip install -U tfx -q


In [ ]:
import os
import tempfile
import urllib.request

import tensorflow as tf
from tensorflow import keras
from tfx import v1 as tfx
from absl import logging

logging.set_verbosity(logging.INFO)

print('TensorFlow version:', tf.__version__)
print('TFX version:', tfx.__version__)


In [ ]:
DATA_ROOT = tempfile.mkdtemp(prefix='tfx-data')
_data_url = ('https://raw.githubusercontent.com/tensorflow/tfx/master/tfx/'
             'examples/penguin/data/labelled/penguins_processed.csv')
_data_filepath = os.path.join(DATA_ROOT, "data.csv")
urllib.request.urlretrieve(_data_url, _data_filepath)

with open(_data_filepath) as f:
    for _ in range(5):
        print(f.readline().strip())


El pipeline base que viste en la clase usaba tres componentes: `CsvExampleGen`, `Trainer` y `Pusher`. Aquí tienes de nuevo la función que lo define, para que puedas reutilizarla en el ejercicio 2:


In [ ]:
def _create_pipeline(pipeline_name, pipeline_root, data_root,
                      module_file, serving_model_dir, metadata_path):
  """Pipeline de referencia: ExampleGen + Trainer + Pusher."""
  example_gen = tfx.components.CsvExampleGen(input_base=data_root)

  trainer = tfx.components.Trainer(
      module_file=module_file,
      examples=example_gen.outputs['examples'],
      train_args=tfx.proto.TrainArgs(num_steps=100),
      eval_args=tfx.proto.EvalArgs(num_steps=5))

  pusher = tfx.components.Pusher(
      model=trainer.outputs['model'],
      push_destination=tfx.proto.PushDestination(
          filesystem=tfx.proto.PushDestination.Filesystem(
              base_directory=serving_model_dir)))

  components = [example_gen, trainer, pusher]

  return tfx.dsl.Pipeline(
      pipeline_name=pipeline_name,
      pipeline_root=pipeline_root,
      metadata_connection_config=tfx.orchestration.metadata
      .sqlite_metadata_connection_config(metadata_path),
      components=components)


---

## Ejercicio 1 — Añade validación de datos con TFDV

Un pipeline sin validación de datos entrena "a ciegas": si algún día llegan datos corruptos o con un tipo equivocado, el `Trainer` los aceptará igual y el problema solo se notará cuando el modelo ya esté fallando en producción.

Tu tarea es completar la función `_create_pipeline_with_validation` de más abajo añadiendo tres componentes de TensorFlow Data Validation (TFDV) al pipeline base: `StatisticsGen`, `SchemaGen` y `ExampleValidator`. El `Trainer` y el `Pusher` no cambian.

**Pistas:**
- `StatisticsGen` solo necesita `examples=example_gen.outputs['examples']`
- `SchemaGen` necesita `statistics=statistics_gen.outputs['statistics']`; usa también `infer_feature_shape=True`
- `ExampleValidator` necesita tanto `statistics=statistics_gen.outputs['statistics']` como `schema=schema_gen.outputs['schema']`
- No olvides añadir los tres componentes nuevos a la lista `components`, **antes** del `trainer`, respetando el orden en que se producen sus artefactos (primero estadísticas, luego esquema, luego validación)


In [ ]:
_trainer_module_file = 'penguin_trainer.py'


In [ ]:
%%writefile {_trainer_module_file}

from typing import List
from absl import logging
import tensorflow as tf
from tensorflow import keras
from tensorflow_transform.tf_metadata import schema_utils

from tfx import v1 as tfx
from tfx_bsl.public import tfxio
from tensorflow_metadata.proto.v0 import schema_pb2

_FEATURE_KEYS = [
    'culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g'
]
_LABEL_KEY = 'species'

_TRAIN_BATCH_SIZE = 20
_EVAL_BATCH_SIZE = 10

_FEATURE_SPEC = {
    **{
        feature: tf.io.FixedLenFeature(shape=[1], dtype=tf.float32)
        for feature in _FEATURE_KEYS
    },
    _LABEL_KEY: tf.io.FixedLenFeature(shape=[1], dtype=tf.int64)
}


def _input_fn(file_pattern: List[str],
              data_accessor: tfx.components.DataAccessor,
              schema: schema_pb2.Schema,
              batch_size: int = 200) -> tf.data.Dataset:
  return data_accessor.tf_dataset_factory(
      file_pattern,
      tfxio.TensorFlowDatasetOptions(
          batch_size=batch_size, label_key=_LABEL_KEY),
      schema=schema).repeat()


def _build_keras_model() -> tf.keras.Model:
  """Crea una red densa (DNN) sencilla para clasificar pinguinos."""
  inputs = [keras.layers.Input(shape=(1,), name=f) for f in _FEATURE_KEYS]
  d = keras.layers.concatenate(inputs)
  for _ in range(2):
    d = keras.layers.Dense(8, activation='relu')(d)
  outputs = keras.layers.Dense(3)(d)

  model = keras.Model(inputs=inputs, outputs=outputs)
  model.compile(
      optimizer=keras.optimizers.Adam(1e-2),
      loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
      metrics=[keras.metrics.SparseCategoricalAccuracy()])

  model.summary(print_fn=logging.info)
  return model


def run_fn(fn_args: tfx.components.FnArgs):
  schema = schema_utils.schema_from_feature_spec(_FEATURE_SPEC)

  train_dataset = _input_fn(
      fn_args.train_files, fn_args.data_accessor, schema,
      batch_size=_TRAIN_BATCH_SIZE)
  eval_dataset = _input_fn(
      fn_args.eval_files, fn_args.data_accessor, schema,
      batch_size=_EVAL_BATCH_SIZE)

  model = _build_keras_model()
  model.fit(
      train_dataset,
      steps_per_epoch=fn_args.train_steps,
      validation_data=eval_dataset,
      validation_steps=fn_args.eval_steps)

  model.save(fn_args.serving_model_dir, save_format='tf')


In [ ]:
def _create_pipeline_with_validation(pipeline_name, pipeline_root, data_root,
                                      module_file, serving_model_dir, metadata_path):
  """Pipeline base + validación de datos con TFDV."""
  example_gen = tfx.components.CsvExampleGen(input_base=data_root)

  # TODO 1: crea un componente StatisticsGen a partir de
  # example_gen.outputs['examples']
  statistics_gen = None  # <- sustituye esta línea

  # TODO 2: crea un componente SchemaGen a partir de
  # statistics_gen.outputs['statistics']. Usa infer_feature_shape=True
  schema_gen = None  # <- sustituye esta línea

  # TODO 3: crea un componente ExampleValidator que reciba las estadísticas
  # de statistics_gen y el esquema de schema_gen
  example_validator = None  # <- sustituye esta línea

  trainer = tfx.components.Trainer(
      module_file=module_file,
      examples=example_gen.outputs['examples'],
      train_args=tfx.proto.TrainArgs(num_steps=100),
      eval_args=tfx.proto.EvalArgs(num_steps=5))

  pusher = tfx.components.Pusher(
      model=trainer.outputs['model'],
      push_destination=tfx.proto.PushDestination(
          filesystem=tfx.proto.PushDestination.Filesystem(
              base_directory=serving_model_dir)))

  # TODO 4: añade statistics_gen, schema_gen y example_validator a esta lista,
  # en el orden correcto y antes del trainer
  components = [
      example_gen,
      trainer,
      pusher,
  ]

  return tfx.dsl.Pipeline(
      pipeline_name=pipeline_name,
      pipeline_root=pipeline_root,
      metadata_connection_config=tfx.orchestration.metadata
      .sqlite_metadata_connection_config(metadata_path),
      components=components)


In [ ]:
PIPELINE_NAME_EX1 = "penguin-practica-tfdv"

tfx.orchestration.LocalDagRunner().run(
    _create_pipeline_with_validation(
        pipeline_name=PIPELINE_NAME_EX1,
        pipeline_root=os.path.join('pipelines', PIPELINE_NAME_EX1),
        data_root=DATA_ROOT,
        module_file=_trainer_module_file,
        serving_model_dir=os.path.join('serving_model', PIPELINE_NAME_EX1),
        metadata_path=os.path.join('metadata', PIPELINE_NAME_EX1, 'metadata.db')))


Si todo salió bien, deberías ver en el registro (log) que `StatisticsGen`, `SchemaGen` y `ExampleValidator` se ejecutan antes que `Trainer`, y `INFO:absl:Component Pusher is finished.` al final.


<details>
<summary>💡 Ver solución</summary>

```python
def _create_pipeline_with_validation(pipeline_name, pipeline_root, data_root,
                                      module_file, serving_model_dir, metadata_path):
  """Pipeline base + validación de datos con TFDV."""
  example_gen = tfx.components.CsvExampleGen(input_base=data_root)

  # Calcula estadísticas descriptivas de cada feature (mínimo, máximo,
  # valores ausentes, histogramas) haciendo una pasada completa sobre los datos.
  statistics_gen = tfx.components.StatisticsGen(
      examples=example_gen.outputs['examples'])

  # Infiere un esquema (tipos de dato, rangos, categorías válidas) a partir
  # de esas estadísticas.
  schema_gen = tfx.components.SchemaGen(
      statistics=statistics_gen.outputs['statistics'],
      infer_feature_shape=True)

  # Compara las estadísticas de los datos contra el esquema inferido y
  # reporta anomalías (valores del tipo equivocado, categorías inesperadas).
  example_validator = tfx.components.ExampleValidator(
      statistics=statistics_gen.outputs['statistics'],
      schema=schema_gen.outputs['schema'])

  trainer = tfx.components.Trainer(
      module_file=module_file,
      examples=example_gen.outputs['examples'],
      train_args=tfx.proto.TrainArgs(num_steps=100),
      eval_args=tfx.proto.EvalArgs(num_steps=5))

  pusher = tfx.components.Pusher(
      model=trainer.outputs['model'],
      push_destination=tfx.proto.PushDestination(
          filesystem=tfx.proto.PushDestination.Filesystem(
              base_directory=serving_model_dir)))

  # El orden importa: los componentes de validación van antes del trainer,
  # porque queremos conocer el estado de los datos antes de entrenar.
  components = [
      example_gen,
      statistics_gen,
      schema_gen,
      example_validator,
      trainer,
      pusher,
  ]

  return tfx.dsl.Pipeline(
      pipeline_name=pipeline_name,
      pipeline_root=pipeline_root,
      metadata_connection_config=tfx.orchestration.metadata
      .sqlite_metadata_connection_config(metadata_path),
      components=components)
```
</details>


---

## Ejercicio 2 — Cambia la arquitectura del modelo

El pipeline no sabe ni le importa qué arquitectura de red usa tu modelo: el `Trainer` simplemente ejecuta la función `run_fn` de tu módulo. Eso significa que puedes experimentar con arquitecturas distintas sin tocar el resto del pipeline.

Tu tarea es completar la función `_build_keras_model` del módulo de más abajo para que la red sea más profunda o tenga regularización, sin cambiar el número de entradas (los cuatro features) ni el de salidas (las tres especies).

**Pistas:**
- Puedes añadir una tercera capa `keras.layers.Dense` (por ejemplo, con 16 neuronas) antes de las dos que ya existen
- Puedes añadir una capa `keras.layers.Dropout(0.2)` entre las capas densas para regularizar el modelo
- Sea lo que sea que cambies, la última capa debe seguir siendo `keras.layers.Dense(3)` porque hay tres especies de pingüinos
- Después de completar el módulo, ejecuta el pipeline de referencia (`_create_pipeline`, definido en la sección de preparación) con este nuevo archivo y compara el resultado en el registro de entrenamiento (busca la línea `sparse_categorical_accuracy`)


In [ ]:
%%writefile {_trainer_module_file}

from typing import List
from absl import logging
import tensorflow as tf
from tensorflow import keras
from tensorflow_transform.tf_metadata import schema_utils

from tfx import v1 as tfx
from tfx_bsl.public import tfxio
from tensorflow_metadata.proto.v0 import schema_pb2

_FEATURE_KEYS = [
    'culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g'
]
_LABEL_KEY = 'species'

_TRAIN_BATCH_SIZE = 20
_EVAL_BATCH_SIZE = 10

_FEATURE_SPEC = {
    **{
        feature: tf.io.FixedLenFeature(shape=[1], dtype=tf.float32)
        for feature in _FEATURE_KEYS
    },
    _LABEL_KEY: tf.io.FixedLenFeature(shape=[1], dtype=tf.int64)
}


def _input_fn(file_pattern: List[str],
              data_accessor: tfx.components.DataAccessor,
              schema: schema_pb2.Schema,
              batch_size: int = 200) -> tf.data.Dataset:
  return data_accessor.tf_dataset_factory(
      file_pattern,
      tfxio.TensorFlowDatasetOptions(
          batch_size=batch_size, label_key=_LABEL_KEY),
      schema=schema).repeat()


def _build_keras_model() -> tf.keras.Model:
  """Crea el modelo Keras que entrena el componente Trainer.

  TODO: cambia esta arquitectura. Por ejemplo, añade una capa Dense
  adicional con más neuronas antes de las dos que ya existen, y/o una
  capa Dropout entre medias para regularizar. No cambies el número de
  entradas (4 features) ni la última capa (Dense(3), una por especie).
  """
  inputs = [keras.layers.Input(shape=(1,), name=f) for f in _FEATURE_KEYS]
  d = keras.layers.concatenate(inputs)

  # TODO: modifica las capas ocultas de aquí abajo.
  for _ in range(2):
    d = keras.layers.Dense(8, activation='relu')(d)

  outputs = keras.layers.Dense(3)(d)  # No cambies esta línea.

  model = keras.Model(inputs=inputs, outputs=outputs)
  model.compile(
      optimizer=keras.optimizers.Adam(1e-2),
      loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
      metrics=[keras.metrics.SparseCategoricalAccuracy()])

  model.summary(print_fn=logging.info)
  return model


def run_fn(fn_args: tfx.components.FnArgs):
  schema = schema_utils.schema_from_feature_spec(_FEATURE_SPEC)

  train_dataset = _input_fn(
      fn_args.train_files, fn_args.data_accessor, schema,
      batch_size=_TRAIN_BATCH_SIZE)
  eval_dataset = _input_fn(
      fn_args.eval_files, fn_args.data_accessor, schema,
      batch_size=_EVAL_BATCH_SIZE)

  model = _build_keras_model()
  model.fit(
      train_dataset,
      steps_per_epoch=fn_args.train_steps,
      validation_data=eval_dataset,
      validation_steps=fn_args.eval_steps)

  model.save(fn_args.serving_model_dir, save_format='tf')


In [ ]:
PIPELINE_NAME_EX2 = "penguin-practica-arquitectura"

tfx.orchestration.LocalDagRunner().run(
    _create_pipeline(
        pipeline_name=PIPELINE_NAME_EX2,
        pipeline_root=os.path.join('pipelines', PIPELINE_NAME_EX2),
        data_root=DATA_ROOT,
        module_file=_trainer_module_file,
        serving_model_dir=os.path.join('serving_model', PIPELINE_NAME_EX2),
        metadata_path=os.path.join('metadata', PIPELINE_NAME_EX2, 'metadata.db')))


<details>
<summary>💡 Ver solución</summary>

```python
def _build_keras_model() -> tf.keras.Model:
  """Crea el modelo Keras que entrena el componente Trainer.

  Versión con una capa oculta adicional (16 neuronas) y una capa Dropout
  para regularizar antes de las dos capas de 8 neuronas originales.
  """
  inputs = [keras.layers.Input(shape=(1,), name=f) for f in _FEATURE_KEYS]
  d = keras.layers.concatenate(inputs)

  # Capa adicional, más ancha, para darle más capacidad al modelo.
  d = keras.layers.Dense(16, activation='relu')(d)
  # Apaga el 20% de las neuronas al azar en cada paso de entrenamiento,
  # para reducir el riesgo de sobreajuste (overfitting).
  d = keras.layers.Dropout(0.2)(d)

  for _ in range(2):
    d = keras.layers.Dense(8, activation='relu')(d)

  outputs = keras.layers.Dense(3)(d)  # Tres especies de pingüinos.

  model = keras.Model(inputs=inputs, outputs=outputs)
  model.compile(
      optimizer=keras.optimizers.Adam(1e-2),
      loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
      metrics=[keras.metrics.SparseCategoricalAccuracy()])

  model.summary(print_fn=logging.info)
  return model
```

El resto del módulo (`_input_fn`, `run_fn`, `_FEATURE_SPEC`) no necesita ningún cambio: el pipeline solo le pide al `Trainer` que ejecute `run_fn`, y `run_fn` solo le pide a `_build_keras_model` un modelo Keras compilado con las entradas y salidas correctas. Con un dataset tan pequeño como este, no siempre notarás una mejora clara de precisión: lo importante es comprobar que puedes cambiar la arquitectura libremente sin tocar ni el pipeline ni el resto del código de entrenamiento.
</details>
